# 6주차 개념 노트북 — 웹 검색 기반 리서치

이 노트북은 4일 치 일별 노트를 하루씩 그대로 반영하며, 실행 가능한 코드를 함께 담고 있습니다.

**이 샌드박스에서 실제로 실행해 검증한 것:** 목업 검색 클라이언트, TF-IDF 재정렬
예시(실제 `scikit-learn` 코사인 유사도 점수), 인용 검증 함수, `research()` 파이프라인,
도구 평가 점수화 함수. 실시간 네트워크를 쓰는 두 셀(1일차의 DuckDuckGo HTML 검색, 3일차의
페이지 가져오기)도 이 환경에서 성공적으로 테스트했지만, 이 노트북을 실행할 때마다 실시간
웹 엔드포인트에 접근 가능하리라고 의존할 수는 없으므로 고정된 폴백과 함께 `try/except`로
감쌌습니다. 이 환경에서 **실행할 수 없는** 유일한 셀은 2일차 끝부분의 `sentence-transformers`
예시입니다 — 이 샌드박스에는 `torch`/`transformers`/`sentence-transformers`가 설치되어 있지
않습니다. 실제 현재 API 기준으로 정확하게 작성했지만 참고용으로만 포함했습니다.

## 1일차: 최신 정보를 위한 검색 API

검색 API는 텍스트 질의를 받아 제목, 스니펫, 출처 URL을 가진 결과 목록을 돌려줍니다. 아래는
작은 **목업** 검색 클라이언트입니다 — 네트워크 호출도, API 키도 없이 — 실제 클라이언트가 가질
형태와 동일합니다. 나중에 목업을 실제 클라이언트로 바꾸는 것은 `search_web`의 본문만 바꾸면
되고, 그것을 호출하는 쪽은 전혀 건드릴 필요가 없습니다.

In [ ]:
from dataclasses import dataclass

@dataclass
class SearchResult:
    title: str
    snippet: str
    source_url: str

# 실제 검색 제공자의 백엔드를 대신하는 작은 로컬 "색인".
# 실제 구현이라면 `query`를 HTTP 엔드포인트로 보내고 JSON 결과를 파싱했을 것이다.
_MOCK_INDEX: dict[str, list[SearchResult]] = {
    "launch window definition": [
        SearchResult(
            "Launch Windows Explained",
            "A launch window is the time period during which a rocket can lift off to reach its target orbit.",
            "https://example-space.org/launch-windows",
        ),
        SearchResult(
            "Orbital Mechanics 101",
            "Launch windows are constrained by the relative positions of Earth and the destination body.",
            "https://example-space.org/orbital-mechanics",
        ),
    ],
    "reusable rocket landing methods": [
        SearchResult(
            "Booster Recovery Techniques",
            "Modern boosters land using either a controlled powered descent or a parachute-assisted splashdown.",
            "https://example-space.org/booster-recovery",
        ),
    ],
}

def search_web_stub(query: str, max_results: int = 5) -> list[SearchResult]:
    """목업 검색 클라이언트. 실제였다면 제공자 API를 호출해 JSON을 SearchResult로 매핑했을 것이다."""
    key = query.lower().strip()
    return _MOCK_INDEX.get(key, [])[:max_results]  # -> list[SearchResult], len 0..max_results

# 구체적인 질의는 목업 색인에 걸리고, 모호한 질의는 쓸모 있는 결과를 하나도 못 돌려준다.
# 실제 검색 엔진이었다면 모호한 질의가 "아무것도 없음"이 아니라, 기술적으로는 주제와
# 관련 있지만 이 구체적인 질문에는 쓸모없는 결과를 잔뜩 돌려줬을 것이다.
good_hits = search_web_stub("launch window definition")
vague_hits = search_web_stub("space stuff")
print(f"specific query -> {len(good_hits)} results")
print(f"vague query    -> {len(vague_hits)} results")

In [ ]:
def build_context_bundle(results: list[SearchResult]) -> str:
    """검색 결과를 LLM 프롬프트용 문자열 하나로 합친다. 각 결과에 번호를 매겨서
    나중에 '[2]' 같은 인용이 results[1]로 곧바로 추적될 수 있도록 한다. 모든 블록에
    URL을 계속 붙여두는 것이 이후(3일차) 인용을 가능하게 만드는 핵심이다."""
    blocks = []
    for i, r in enumerate(results, start=1):
        blocks.append(f"[{i}] {r.title}\n{r.snippet}\nSource: {r.source_url}")
    return "\n\n".join(blocks)  # -> str, SearchResult 하나당 빈 줄로 구분된 블록 하나

bundle = build_context_bundle(good_hits)
print(bundle)

### 비교를 위한 실제 호출

지금까지는 전부 결정론적이고 오프라인입니다. 아래는 DuckDuckGo의 공개 HTML 결과 페이지에
실제 HTTP 요청을 보내(API 키 없이) 얻은, 같은 `SearchResult` 형태입니다 — 어떤 유료 검색
API든 내부적으로 하고 있는 실제 요청/파싱 메커니즘을 보는 데 유용합니다. 네트워크가 없어도
이 셀이 실행되도록 고정된 폴백과 함께 `try/except`로 감쌌습니다.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote

def search_web_live(query: str, max_results: int = 5) -> list[SearchResult]:
    """어디까지나 설명용 -- 문서화되지 않은 HTML 엔드포인트를 API 키 없이 호출하므로
    취약하고(마크업이 바뀔 수 있음) 프로덕션에는 적합하지 않다. 실제 통합이라면 HTML을
    스크래핑하는 대신 문서화되고 이용약관을 준수하는 제공자(Tavily, Brave Search API,
    Bing Web Search, Serper 등)를 정식 키로 호출해야 한다."""
    resp = requests.get(
        "https://html.duckduckgo.com/html/",
        params={"q": query},
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results: list[SearchResult] = []
    for row in soup.select(".result")[:max_results]:
        link = row.select_one(".result__title a")
        snippet_el = row.select_one(".result__snippet")
        if link is None:
            continue
        # 이 HTML 엔드포인트는 실제 목적지를 바로 링크하는 대신
        # `/l/?uddg=<url-encoded-url>` 형태의 리다이렉트 링크로 감싼다 --
        # 나중에 인용이 실제 위치를 가리키도록 이를 디코딩해서 실제 source_url을 복원한다.
        qs = parse_qs(urlparse(link.get("href", "")).query)
        real_url = unquote(qs["uddg"][0]) if "uddg" in qs else link.get("href", "")
        results.append(SearchResult(
            title=link.get_text(strip=True),
            snippet=snippet_el.get_text(strip=True) if snippet_el else "",
            source_url=real_url,
        ))
    return results  # -> list[SearchResult], len <= max_results

# 고정된 폴백 데이터: search_web_live("python asyncio wait_for timeout")를 캡처했을 때
# 실제로 돌아온 결과 그대로 -- 오프라인에서도 이 셀이 같은 형태를 보여주도록 한다.
_LIVE_SEARCH_FALLBACK = [
    SearchResult(
        "Asyncio wait_for() to Wait With a Timeout - SuperFastPython",
        "",
        "https://superfastpython.com/asyncio-wait_for/",
    ),
    SearchResult(
        "Python asyncio.wait_for(): Cancel a Task with a Timeout",
        "",
        "https://www.pythontutorial.net/python-concurrency/python-asyncio-wait_for/",
    ),
]

try:
    live_hits = search_web_live("python asyncio wait_for timeout", max_results=5)
    if not live_hits:
        raise ValueError("live search returned no results")
    source_label = "LIVE"
except Exception as exc:
    live_hits = _LIVE_SEARCH_FALLBACK
    source_label = f"FALLBACK (live call unavailable: {type(exc).__name__})"

print(f"[{source_label}] {len(live_hits)} results")
for r in live_hits[:3]:
    print(f"  - {r.title}  |  {r.source_url}")

## 2일차: 임베딩 기반 검색 결과 재정렬

검색 API 자체의 순위는 에이전트가 지금 물어본 정확한 질의가 아니라 일반적인 인기도에 맞춰
최적화되어 있습니다. 재정렬은 임베딩된 텍스트에 대한 코사인 유사도로 원시 결과를 그 특정
질의에 맞춰 다시 점수 매기므로, 진짜 관련 있는 결과만 LLM에 도달합니다.

In [ ]:
import hashlib
import re
from collections import Counter

VECTOR_SIZE = 64

def embed_text(text: str) -> list[float]:
    """실제 임베딩 모델 호출을 대신하는, 결정론적이고 완전히 로컬인 대체물. 모든 단어는
    매번 같은 버킷으로 해시되므로 동일한 텍스트는 항상 동일한 벡터를 만든다 -- 모델
    가중치도 무작위성도 전혀 개입하지 않는다. 이는 공유된 단어를 통해서만 관련성을
    포착하는 희소(sparse)한, 어휘적 표현이다."""
    vector = [0.0] * VECTOR_SIZE          # -> list[float], len 64, 전부 0으로 시작
    words = re.findall(r"[a-z0-9]+", text.lower())
    for word, count in Counter(words).items():
        bucket = int(hashlib.md5(word.encode()).hexdigest(), 16) % VECTOR_SIZE
        vector[bucket] += count            # 충돌이 나면 그냥 같은 버킷에 개수가 더해질 뿐이다
    return vector

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0  # -> [-1.0, 1.0] 사이의 float

def rerank_results(query: str, results: list[SearchResult], top_k: int = 2) -> list[SearchResult]:
    """각 SearchResult(title + snippet)를 질의와 비교해 점수를 매긴 다음, 가장 관련성
    높은 top_k개만 남긴다 -- 나머지는 LLM의 컨텍스트에 절대 들어가지 않는다."""
    query_vec = embed_text(query)
    scored = [
        (cosine_similarity(query_vec, embed_text(r.title + " " + r.snippet)), r)
        for r in results
    ]  # -> list[tuple[float, SearchResult]], results와 길이가 같음
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [r for _, r in scored[:top_k]]  # -> list[SearchResult], len == top_k

In [ ]:
query = "starter hydration ratio"

# 검색 API가 돌려줄 법한 원시 결과: 관련성과 인기도가 뒤섞여 있다.
raw_results = [
    SearchResult("History of Sourdough Bread", "This ancient bread-making technique dates back thousands of years to Ancient Egypt.", "https://example-bake.org/history"),
    SearchResult("Hydration Ratio for Sourdough Starter", "A 100 percent hydration starter uses equal weights of flour and water by mass.", "https://example-bake.org/hydration"),
    SearchResult("Best Bread Knives 2026", "A serrated knife makes cleaner slices through a crusty loaf without tearing it.", "https://example-bake.org/knives"),
    SearchResult("Adjusting Starter Hydration for Climate", "Lower hydration starter mixtures ferment more slowly in humid kitchens.", "https://example-bake.org/climate-hydration"),
    SearchResult("Sourdough Discard Recipes", "Use leftover starter portions in pancakes or crackers instead of discarding them.", "https://example-bake.org/discard"),
]

print("BEFORE (raw API order):")
for r in raw_results:
    print(f"  - {r.title}")

print("\nAFTER (hash-embedding re-ranked, top 2):")
for r in rerank_results(query, raw_results, top_k=2):
    print(f"  - {r.title}")

### 실제 검증 가능한 점수로 보는 같은 재정렬 (scikit-learn의 TF-IDF)

위 해시 버킷 임베딩은 의도적으로 최소한의 형태입니다. `TfidfVectorizer`는 진짜로 널리 쓰이는
희소 임베딩 방식이며, 순서만이 아니라 실제 코사인 유사도 숫자를 출력할 수 있게 해줍니다.
검증된 출력은 셀 뒤에 주석으로 표시했습니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity
import numpy as np

titles = [r.title for r in raw_results]
docs = [f"{r.title}. {r.snippet}" for r in raw_results]

# fit_transform은 말뭉치 어휘를 학습함과 동시에 모든 문서를 한 번에 인코딩한다.
# shape: (n_docs=5, vocab_size) -- vocab_size는 말뭉치 텍스트에 따라 달라진다는 점이
# 위 해시 기반 embed_text가 쓰는 고정값 VECTOR_SIZE=64와 다르다.
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(docs)

# fit_transform이 아니라 transform을 쓰는 이유: 질의에도 같은 학습된 어휘를 재사용해야
# 질의와 문서가 정확히 같은 벡터 공간에 놓여 직접 비교 가능해지기 때문이다.
# shape: (1, vocab_size)
query_vec = vectorizer.transform([query])

sims = sk_cosine_similarity(query_vec, doc_matrix)[0]  # -> np.ndarray, shape (5,), 문서당 점수 하나
order = np.argsort(-sims)                               # 관련성 내림차순

print(f"doc_matrix.shape = {doc_matrix.shape}, query_vec.shape = {query_vec.shape}")
print("\nAFTER (TF-IDF cosine rerank, real scores):")
for idx in order:
    print(f"  {sims[idx]:.4f}  {titles[idx]}")

# 검증된 출력 (이 셀은 이 샌드박스에서 실제로 실행했다):
#   doc_matrix.shape = (5, 49), query_vec.shape = (1, 49)
#   0.5933  Hydration Ratio for Sourdough Starter
#   0.4310  Adjusting Starter Hydration for Climate
#   0.0984  Sourdough Discard Recipes
#   0.0000  History of Sourdough Bread
#   0.0000  Best Bread Knives 2026
# 주제와 관련 있는 두 결과가 가장 높은 점수를 받고, 빵칼 결과와 역사 결과는 정확히
# 0.0점이다 -- 불용어를 제거하고 나면 질의와 어휘를 전혀 공유하지 않기 때문이다.
# 전혀 겹치지 않는 벡터 사이의 코사인 유사도는 "낮음"이 아니라 수학적으로 정확히 0이다.

### 밀집 임베딩 (이 샌드박스에서는 실행 불가)

TF-IDF는 여전히 어휘적입니다: 단어를 하나도 공유하지 않는 "starter hydration ratio"와
"flour-to-water proportions"라는 표현을 연결하지 못합니다. 실제 밀집(dense) 임베딩 모델은
토큰을 세는 대신 의미를 인코딩해서 그 간극을 메웁니다. 이 셀은 현재 실제
`sentence-transformers` API 기준으로 작성했지만 여기서는 **실행하지 않습니다** --
이 샌드박스에는 `torch`/`transformers`/`sentence-transformers`가 설치되어 있지 않습니다
(이 노트북 맨 위의 안내 참고).

In [ ]:
# 이 샌드박스에서는 실행되지 않음 -- 이 노트북 맨 위의 안내 참고.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # 사전학습된 소형 트랜스포머를 한 번 로드한다

def embed_text_dense(texts: list[str]) -> np.ndarray:
    # normalize_embeddings=True는 단위 벡터를 반환하므로, 이후 단순 내적만으로도
    # 이미 코사인 유사도와 같아진다 -- 별도로 norm으로 나눌 필요가 없다.
    return model.encode(texts, normalize_embeddings=True)  # -> shape (len(texts), 384)

# query_vec = embed_text_dense([query])[0]           # shape (384,)
# doc_vecs = embed_text_dense(docs)                    # shape (5, 384)
# dense_sims = doc_vecs @ query_vec                    # 단위 벡터 -> 내적 == 코사인 유사도

## 3일차: 검색 + LLM 근거 기반 리서치

근거 부여란 LLM이 검색된 자료에서만 답하고, 각 주장을 어떤 출처가 뒷받침하는지 인용하고,
자료가 불충분할 때는 추측하는 대신 그렇다고 인정하는 것을 뜻합니다. 아래 파이프라인은 검색
단계, 재정렬 단계, (목업) 근거 기반 LLM 호출을 연결합니다.

In [ ]:
def build_grounded_prompt(question: str, bundle: str) -> str:
    return (
        "You are a research assistant. Answer using ONLY the material below.\n\n"
        f"MATERIAL:\n{bundle}\n\n"
        f"QUESTION: {question}\n\n"
        "Rules:\n"
        "- Cite the source number [n] after every factual claim.\n"
        "- If the material does not answer the question, say "
        "'Not enough information in the provided sources.'\n"
        "- Do not add outside knowledge.\n"
    )

def call_llm_stub(prompt: str) -> str:
    """목업 LLM 호출. 실제 클라이언트(예: Anthropic Messages API 호출)라면 이 고정된
    응답 대신 `prompt`를 호스팅된 모델로 보내고 그 텍스트 응답을 돌려받았을 것이다."""
    return (
        "QUIC is now supported by default in several major browsers [1]. Adoption on the server "
        "side is growing but remains behind HTTP/2 in raw traffic share [2]. Not enough information "
        "in the provided sources to say when server-side adoption will overtake HTTP/2.\n\n"
        "Sources:\n[1] https://example-net.org/quic-browsers\n[2] https://example-net.org/quic-traffic-share"
    )

In [ ]:
protocol_index = {
    "quic protocol adoption": [
        SearchResult(
            "QUIC Support Across Browsers",
            "QUIC is enabled by default in several major browsers as of this year.",
            "https://example-net.org/quic-browsers",
        ),
        SearchResult(
            "Server-Side Traffic Share Report",
            "QUIC traffic is rising but still trails HTTP/2 in overall share among measured servers.",
            "https://example-net.org/quic-traffic-share",
        ),
    ],
}

def search_web_v2(query: str, max_results: int = 5) -> list[SearchResult]:
    return protocol_index.get(query.lower().strip(), [])[:max_results]

def research(question: str, searcher, ranker, summarizer) -> str:
    """연결: 검색 -> 재정렬 -> 출처 포함 번들링 -> 근거 기반 LLM 요약.
    searcher/ranker/summarizer를 (하드코딩하지 않고) 인자로 넘기므로, 이 함수의
    본문을 전혀 건드리지 않고도 목업 스텁이나 실제 클라이언트를 교체할 수 있다."""
    raw_hits = searcher(question)                                        # -> list[SearchResult]
    top_hits = ranker(question, raw_hits, top_k=2) if raw_hits else raw_hits  # -> list[SearchResult]
    bundle = build_context_bundle(top_hits)                                # -> str
    prompt = build_grounded_prompt(question, bundle)                        # -> str
    return summarizer(prompt)                                              # -> str

answer = research(
    "quic protocol adoption",
    searcher=search_web_v2,
    ranker=rerank_results,
    summarizer=call_llm_stub,
)
print(answer)

### 코드로 인용 검증하기

프롬프트는 부탁할 뿐이고, 이 함수는 실제로 확인합니다. 두 가지 구체적인 실패 양상을 잡아냅니다:
**인용 없는 주장**(`[n]` 표시가 없는 문장)과 **환각 인용**(실제로 검색된 출처 개수를 넘어서는
`[n]`).

In [ ]:
import re

REFUSAL_PHRASE = "not enough information in the provided sources"

def split_claim_sentences(answer_body: str) -> list[str]:
    """문장 끝 구두점 기준으로 나누는 단순한 분리 -- 짧은 근거 기반 답변에는 충분하며,
    프로덕션 버전이라면 'e.g.' 같은 예외 케이스를 위해 진짜 문장 토크나이저를 쓸 것이다."""
    raw = re.split(r"(?<=[.!?])\s+", answer_body.strip())
    return [s.strip() for s in raw if s.strip()]  # -> list[str], 문장마다 하나씩

def validate_citations(answer: str, num_sources: int) -> dict:
    """예외를 던지는 대신 보고서 형태의 dict를 반환해서, 호출자가 얼마나 엄격하게
    다룰지 결정할 수 있게 한다(예: 인용 없는 문장은 조용히 버릴지, 답변 전체를
    거부하고 재시도할지)."""
    body = answer.split("Sources:")[0].strip()
    lower_body = body.lower()

    # 인용을 생략해도 되는 유일하게 명시적으로 허용된 문장: 근거 기반 거절 그 자체.
    if REFUSAL_PHRASE in lower_body and len(split_claim_sentences(body)) == 1:
        return {"ok": True, "uncited_claims": [], "out_of_range_citations": []}

    sentences = split_claim_sentences(body)                          # -> list[str], len = 주장 개수
    uncited = [s for s in sentences if not re.search(r"\[\d+\]", s)]  # -> list[str], 실패 양상 1

    cited_numbers = {int(n) for n in re.findall(r"\[(\d+)\]", body)}  # -> set[int]
    out_of_range = sorted(n for n in cited_numbers if n < 1 or n > num_sources)  # 실패 양상 2

    return {
        "ok": not uncited and not out_of_range,
        "uncited_claims": uncited,
        "out_of_range_citations": out_of_range,
    }

# --- 테스트 케이스 ---
good = (
    "The library added native retry support in v2.3 [1]. Backoff intervals are "
    "configurable via a backoff_factor argument [2].\n\n"
    "Sources:\n[1] https://example.dev/changelog/v2.3\n[2] https://example.dev/docs/retries"
)
hallucinated_citation = (
    "The library added native retry support in v2.3 [1]. It also supports circuit "
    "breakers out of the box [3].\n\n"
    "Sources:\n[1] https://example.dev/changelog/v2.3\n[2] https://example.dev/docs/retries"
)
uncited_claim = (
    "The library added native retry support in v2.3 [1]. It is the fastest retry "
    "library available today.\n\n"
    "Sources:\n[1] https://example.dev/changelog/v2.3\n[2] https://example.dev/docs/retries"
)
refusal = "Not enough information in the provided sources.\n\nSources:\n[1] https://example.dev/changelog/v2.3"

for name, text, n in [
    ("good", good, 2),
    ("hallucinated_citation", hallucinated_citation, 2),
    ("uncited_claim", uncited_claim, 2),
    ("refusal", refusal, 1),
]:
    print(name, "->", validate_citations(text, n))

# 검증된 출력 (이 샌드박스에서 실행함):
# good                   -> {'ok': True,  'uncited_claims': [], 'out_of_range_citations': []}
# hallucinated_citation  -> {'ok': False, 'uncited_claims': [], 'out_of_range_citations': [3]}
# uncited_claim          -> {'ok': False, 'uncited_claims': ['It is the fastest retry library available today.'], 'out_of_range_citations': []}
# refusal                -> {'ok': True,  'uncited_claims': [], 'out_of_range_citations': []}

### 스니펫 대 전체 페이지

검색 스니펫은 흔히 제공자의 하이라이팅 로직이 고른 조각일 뿐, 질문에 답하는 문장이라는
보장은 없습니다. 전체 페이지를 가져오면 스니펫에서 잘려나간 세부 내용을 복원할 수 있습니다.
안정적인 대상(파이썬 공식 문서)을 대상으로 테스트했으며, 오프라인에서도 셀이 실행되도록
고정된 폴백 문자열과 함께 `try/except`로 감쌌습니다.

In [ ]:
def fetch_page_text(url: str, timeout: float = 10.0) -> str:
    """페이지의 보이는 문단 텍스트를 가져와 평탄화한다. 네트워크 오류가 나면 빈
    문자열로 폴백해서, 가져오기 실패가 파이프라인 전체를 죽이는 대신 '스니펫만
    사용'으로 우아하게 저하되게 한다 -- 1일차 목업 검색 클라이언트와 같은 철학이다."""
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=timeout)
        resp.raise_for_status()
    except requests.RequestException:
        return ""  # 호출자는 이를 "전체 페이지 텍스트를 사용할 수 없음"과 동일하게 취급해야 한다
    soup = BeautifulSoup(resp.text, "html.parser")
    main = soup.find("main") or soup
    paragraphs = main.find_all("p")
    return " ".join(p.get_text(" ", strip=True) for p in paragraphs)  # -> str, 평탄화된 본문 텍스트

_FETCH_FALLBACK = (
    "...cancelled. Example: Changed in version 3.7: When aw is cancelled due to a timeout, "
    "wait_for waits for aw to be cancelled. Previously, it raised TimeoutError immediately. "
    "Changed in version 3.10: Removed the loop parameter. Changed in version 3.11: Raises "
    "TimeoutError instead of asyncio.TimeoutError. Changed in version 3.12: Implemented using "
    "asyncio.timeout()..."
)

page_text = fetch_page_text("https://docs.python.org/3/library/asyncio-task.html")
if not page_text:
    page_text = _FETCH_FALLBACK
    print("[FALLBACK -- 네트워크 사용 불가, 캐시된 스니펫 사용]")

idx = page_text.lower().find("wait_for")
print(page_text[max(0, idx - 40): idx + 220] if idx >= 0 else page_text[:260])

## 4일차: AI 도구 평가 프레임워크

이 분야의 특정 도구 이름은 금방 낡아버리므로, 평가 자체는 프레임워크와 무관하게 유지합니다:
후보 도구를 비용, 보안, 승인 난이도로 점수화하고, 3일차의 `research()` 파이프라인을 재사용해
그 근거가 되는 사실을 최신 상태로 유지합니다.

In [ ]:
from dataclasses import dataclass

@dataclass
class ToolEvaluation:
    name: str
    pricing_model: str
    hidden_usage_costs: bool
    trains_on_input_data: bool
    sso_supported: bool
    requires_security_review: bool
    notes: str = ""

    def risk_score(self) -> int:
        """가산식 위험 점수, 0 = 가장 낮은 위험. 가중치(1~2)는 의도적으로 거칠다 --
        요점은 소수점 두 자리까지 논쟁할 정밀한 숫자를 만드는 게 아니라, '명백히
        괜찮음'과 '명백히 그렇지 않음'을 빠르게 구분하는 것이다."""
        score = 0
        score += 2 if self.hidden_usage_costs else 0          # 예산을 조용히 초과시킬 수 있음
        score += 2 if self.trains_on_input_data else 0        # 당신의 데이터가 통제권을 벗어남
        score += 1 if not self.sso_supported else 0           # 인증/오프보딩이 더 취약함
        score += 1 if self.requires_security_review else 0    # 위험이 아니라 마찰이지만 도입을 늦춤
        return score

    def quick_verdict(self) -> str:
        score = self.risk_score()
        if score == 0:
            return "green"
        if score >= 4:
            return "red"
        return "yellow"

candidates = [
    ToolEvaluation(
        name="note-taking assistant",
        pricing_model="per-seat monthly",
        hidden_usage_costs=False,
        trains_on_input_data=False,
        sso_supported=True,
        requires_security_review=False,
        notes="Opt-out training disclosed clearly in settings.",
    ),
    ToolEvaluation(
        name="spreadsheet copilot",
        pricing_model="usage-metered, billed to a connected API key",
        hidden_usage_costs=True,
        trains_on_input_data=True,
        sso_supported=False,
        requires_security_review=True,
        notes="Retention policy unclear; needs compliance sign-off before rollout.",
    ),
    ToolEvaluation(
        name="code review bot",
        pricing_model="free tier, usage-metered beyond 500 reviews/month",
        hidden_usage_costs=False,
        trains_on_input_data=True,
        sso_supported=True,
        requires_security_review=True,
        notes="Trains on input by default but SSO + opt-out available on request.",
    ),
]

for c in candidates:
    print(f"{c.name}: risk_score={c.risk_score()} verdict={c.quick_verdict()} | {c.notes}")

# 검증된 출력 (이 샌드박스에서 실행함):
# note-taking assistant: risk_score=0 verdict=green | Opt-out training disclosed clearly in settings.
# spreadsheet copilot: risk_score=6 verdict=red | Retention policy unclear; needs compliance sign-off before rollout.
# code review bot: risk_score=3 verdict=yellow | Trains on input by default but SSO + opt-out available on request.

In [ ]:
# 3일차와 연결: 위의 비용/보안/승인-난이도 점수가 낡지 않도록, 특정 도구 카테고리에
# 무엇이 바뀌었는지 주기적으로 리서치한다.
landscape_index = {
    "spreadsheet copilot pricing and security changes": [
        SearchResult(
            "Spreadsheet Copilot Updates Data Retention Policy",
            "The vendor now offers an enterprise tier with opt-out training and a signed DPA.",
            "https://example-tools.org/copilot-policy-update",
        ),
    ],
}

def search_web_v3(query: str, max_results: int = 5) -> list[SearchResult]:
    return landscape_index.get(query.lower().strip(), [])[:max_results]

landscape_update = research(
    "spreadsheet copilot pricing and security changes",
    searcher=search_web_v3,
    ranker=rerank_results,
    summarizer=call_llm_stub,
)
print(landscape_update)